# RL Run Analysis

This notebook analyzes W&B runs from RL training experiments (using `hf_trainer` with GRPOTrainer).
Fetches and visualizes reward metrics, parsing statistics, training progress, and sample outputs.

**Setup:** Ensure wandb is configured (`wandb login` or API key in `.env`).

In [ ]:
import json
import typing

import IPython.display as ipy_display
import matplotlib.figure
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import notebooks.notebook_utils as nb_utils
import pyine.utils.reprod

In [ ]:
pyine.utils.reprod.entrypoint_setup()
nb_utils.setup_notebook_plotting(use_seaborn=True, seaborn_style="whitegrid")

# ------------ CONFIGURATION ------------
WANDB_PROJECT = "pyine-tests"  # change to your project
WANDB_ENTITY = None  # optional: your team/user entity

# option 1: specify run by URL
WANDB_RUN_URL = ""  # e.g., "https://wandb.ai/entity/project/runs/abc123"

# option 2: specify run by ID
WANDB_RUN_ID = ""  # e.g., "abc123"

# optional: parquet cache path for large runs (set to None to disable)
HISTORY_CACHE_PATH = None  # e.g., "/tmp/wandb_history_cache.parquet"

In [ ]:
run = nb_utils.get_wandb_run(WANDB_RUN_URL, WANDB_RUN_ID, WANDB_PROJECT, WANDB_ENTITY)
print(f"Loaded run: {run.name} ({run.url})")
print(f"State: {run.state}, Created: {run.created_at}")

In [ ]:
# display key summary metrics and configuration
summary = dict(run.summary)
config = dict(run.config)

print("Run Configuration:")
config_keys_of_interest = [
    "model_name",
    "base_model",
    "dataset",
    "num_train_epochs",
    "per_device_train_batch_size",
    "learning_rate",
    "num_generations",
]
for key in config_keys_of_interest:
    if key in config:
        print(f"  {key}: {config[key]}")

# show nested config keys if present
if "grpo_config" in config:
    print("  grpo_config:")
    grpo = config["grpo_config"]
    for subkey in ["num_generations", "max_completion_length", "temperature", "beta"]:
        if subkey in grpo:
            print(f"    {subkey}: {grpo[subkey]}")

print("\nRun Summary (key metrics):")
summary_keys_of_interest = [
    "reward/run/total/mean",
    "reward/run/total/sample_count",
    "train/global_step",
    "_step",
]
for key in summary_keys_of_interest:
    if key in summary:
        print(f"  {key}: {summary[key]}")

In [ ]:
available_keys = nb_utils.discover_wandb_metric_keys(run)

print("Available Metrics in This Run:")
for category, keys in available_keys.items():
    if keys:
        print(f"\n  {category} ({len(keys)} keys):")
        for key in keys[:10]:
            print(f"    - {key}")
        if len(keys) > 10:
            print(f"    ... and {len(keys) - 10} more")
    else:
        print(f"\n  {category}: (none)")

In [ ]:
# fetch important metrics
important_keys = (
    available_keys["reward_total"]
    + available_keys["reward_terms"]
    + available_keys["parsing"][:10]  # limit parsing keys
    + available_keys["trl"][:10]  # limit TRL keys
    + available_keys["step_keys"]
)
# add _step if not already present
if "_step" not in important_keys:
    important_keys.append("_step")

history_df = nb_utils.fetch_wandb_history_df(
    run,
    keys=important_keys if important_keys else None,
    cache_path=HISTORY_CACHE_PATH,
)
step_key = nb_utils.resolve_wandb_step_key(history_df)
time_key = nb_utils.resolve_wandb_time_key(history_df)

print(f"\nFetched {len(history_df)} history rows with {len(history_df.columns)} columns")
print(f"Using step key: {step_key}")
if time_key:
    print(f"Time key available: {time_key}")

## Reward Metrics

In [ ]:
def plot_reward_over_time(
    history_df: pd.DataFrame,
    reward_key: str = "reward/total",
    step_key: str = "_step",
    title: str = "Total Reward Over Training",
    figsize: tuple[int, int] = (12, 5),
    rolling_window: int = 50,
) -> matplotlib.figure.Figure:
    """Plot reward metric over training steps with rolling average."""
    fig, ax = plt.subplots(figsize=figsize)
    if reward_key not in history_df.columns:
        ax.text(
            0.5,
            0.5,
            f"Metric '{reward_key}' not logged in this run",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=12,
        )
        ax.set_title(title)
        return fig
    valid_df = history_df[[step_key, reward_key]].dropna()
    if len(valid_df) == 0:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center", transform=ax.transAxes)
        return fig
    x = valid_df[step_key].values
    y = valid_df[reward_key].values
    ax.scatter(x, y, alpha=0.3, s=10, label="Raw values", color="#2C7BB6")
    if len(y) > rolling_window:
        rolling_mean = pd.Series(y).rolling(window=rolling_window, center=True).mean()
        rolling_std = pd.Series(y).rolling(window=rolling_window, center=True).std()
        ax.plot(x, rolling_mean, color="#D7191C", linewidth=2, label=f"Rolling mean (w={rolling_window})")
        ax.fill_between(
            x,
            rolling_mean - rolling_std,
            rolling_mean + rolling_std,
            alpha=0.2,
            color="#D7191C",
            label="Rolling std",
        )
    ax.set_xlabel(step_key.replace("_", " ").title())
    ax.set_ylabel("Reward")
    ax.set_title(title)
    ax.legend(loc="best", fontsize=9)
    ax.grid(True, alpha=0.3)
    return fig


fig = plot_reward_over_time(history_df, step_key=step_key)
plt.tight_layout()
plt.show()

In [ ]:
def plot_reward_terms_over_time(
    history_df: pd.DataFrame,
    term_keys: list[str],
    step_key: str = "_step",
    figsize: tuple[int, int] = (14, 6),
    rolling_window: int = 50,
) -> matplotlib.figure.Figure:
    """Plot all reward terms over time on the same axes."""
    fig, ax = plt.subplots(figsize=figsize)
    if not term_keys:
        ax.text(
            0.5,
            0.5,
            "No reward term keys found",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        return fig
    colors = plt.cm.tab10.colors
    for idx, key in enumerate(term_keys):
        if key not in history_df.columns:
            continue
        valid_df = history_df[[step_key, key]].dropna()
        if len(valid_df) == 0:
            continue
        x = valid_df[step_key].values
        y = valid_df[key].values
        term_name = key.replace("reward/terms/", "")
        color = colors[idx % len(colors)]
        if len(y) > rolling_window:
            rolling_mean = pd.Series(y).rolling(window=rolling_window, center=True).mean()
            ax.plot(x, rolling_mean, color=color, linewidth=2, label=term_name)
        else:
            ax.plot(x, y, color=color, linewidth=1, alpha=0.7, label=term_name)
    ax.set_xlabel(step_key.replace("_", " ").title())
    ax.set_ylabel("Reward Value")
    ax.set_title("Reward Terms Over Training")
    ax.legend(loc="best", fontsize=9)
    ax.grid(True, alpha=0.3)
    return fig


fig = plot_reward_terms_over_time(history_df, available_keys["reward_terms"], step_key=step_key)
plt.tight_layout()
plt.show()

In [ ]:
def extract_term_stats(
    summary: dict[str, typing.Any],
    term_prefix: str = "reward/run/terms/",
) -> dict[str, dict[str, float]]:
    """Extract {term_name: {mean, std, min, max, sample_count}} from summary."""
    result: dict[str, dict[str, float]] = {}
    for key, value in summary.items():
        if not key.startswith(term_prefix):
            continue
        parts = key[len(term_prefix) :].split("/")
        if len(parts) >= 2:
            term_name, stat = parts[0], parts[1]
            if term_name not in result:
                result[term_name] = {}
            if isinstance(value, (int, float)):
                result[term_name][stat] = float(value)
    return result


def plot_reward_terms_summary(
    summary: dict[str, typing.Any],
    term_prefix: str = "reward/run/terms/",
    figsize: tuple[int, int] = (12, 5),
) -> matplotlib.figure.Figure:
    """Bar chart comparing final reward term statistics."""
    fig, ax = plt.subplots(figsize=figsize)
    term_stats = extract_term_stats(summary, term_prefix)
    if not term_stats:
        ax.text(
            0.5,
            0.5,
            "No reward term summary statistics found",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        ax.set_title("Reward Terms Summary")
        return fig
    term_names = list(term_stats.keys())
    means = [term_stats[t].get("mean", 0) for t in term_names]
    stds = [term_stats[t].get("std", 0) for t in term_names]
    x = np.arange(len(term_names))
    bars = ax.bar(x, means, yerr=stds, capsize=5, color=plt.cm.tab10.colors[: len(term_names)], alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(term_names, rotation=45, ha="right")
    ax.set_ylabel("Mean Reward")
    ax.set_title("Reward Terms Summary (with std error bars)")
    ax.grid(axis="y", alpha=0.3)
    for bar_idx, bar in enumerate(bars):
        height = bar.get_height()
        count = term_stats[term_names[bar_idx]].get("sample_count", None)
        label = f"{height:.3f}"
        if count:
            label += f"\nn={int(count)}"
        ax.annotate(
            label,
            xy=(bar.get_x() + bar.get_width() / 2, height + stds[bar_idx]),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
        )
    return fig


fig = plot_reward_terms_summary(summary)
plt.tight_layout()
plt.show()

In [ ]:
def extract_category_stats(
    summary: dict[str, typing.Any],
    category_prefix: str = "reward/run/categories/",
) -> dict[str, dict[str, float]]:
    """Extract {category: {mean, std, min, max, sample_count}} from summary."""
    result: dict[str, dict[str, float]] = {}
    for key, value in summary.items():
        if not key.startswith(category_prefix):
            continue
        parts = key[len(category_prefix) :].split("/")
        if len(parts) >= 2:
            category, stat = parts[0], parts[1]
            if category not in result:
                result[category] = {}
            if isinstance(value, (int, float)):
                result[category][stat] = float(value)
    return result


def plot_category_rewards(
    summary: dict[str, typing.Any],
    category_prefix: str = "reward/run/categories/",
    figsize: tuple[int, int] = (14, 6),
) -> matplotlib.figure.Figure:
    """Bar chart of rewards by category."""
    fig, ax = plt.subplots(figsize=figsize)
    cat_stats = extract_category_stats(summary, category_prefix)
    if not cat_stats:
        ax.text(
            0.5,
            0.5,
            "No category reward statistics found",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        ax.set_title("Rewards by Category")
        return fig
    categories = list(cat_stats.keys())
    means = [cat_stats[c].get("mean", 0) for c in categories]
    stds = [cat_stats[c].get("std", 0) for c in categories]
    counts = [cat_stats[c].get("sample_count", 0) for c in categories]
    x = np.arange(len(categories))
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(categories)))
    bars = ax.bar(x, means, yerr=stds, capsize=4, color=colors, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(categories, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("Mean Reward")
    ax.set_title("Rewards by Category (with std error bars)")
    ax.grid(axis="y", alpha=0.3)
    for bar_idx, bar in enumerate(bars):
        height = bar.get_height()
        count = counts[bar_idx]
        label_y = height + stds[bar_idx] if stds[bar_idx] else height
        ax.annotate(
            f"n={int(count)}",
            xy=(bar.get_x() + bar.get_width() / 2, label_y),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=7,
        )
    return fig


fig = plot_category_rewards(summary)
plt.tight_layout()
plt.show()

## Parsing Statistics

In [ ]:
def plot_parsing_stats(
    history_df: pd.DataFrame,
    parsing_keys: list[str],
    step_key: str = "_step",
    figsize: tuple[int, int] = (16, 10),
    rolling_window: int = 50,
) -> matplotlib.figure.Figure:
    """Plot parsing-related metrics over time."""
    # group keys by type
    length_keys = [k for k in parsing_keys if "length" in k]
    ratio_keys = [k for k in parsing_keys if "has_" in k or "missing" in k or "malformed" in k]
    other_keys = [k for k in parsing_keys if k not in length_keys and k not in ratio_keys]
    nrows = sum(1 for group in [length_keys, ratio_keys, other_keys] if group)
    if nrows == 0:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.text(
            0.5,
            0.5,
            "No parsing metrics found",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        return fig
    fig, axes = plt.subplots(nrows, 1, figsize=figsize)
    if nrows == 1:
        axes = [axes]
    ax_idx = 0
    colors = plt.cm.tab10.colors
    # plot length metrics
    if length_keys:
        ax = axes[ax_idx]
        for key_idx, key in enumerate(length_keys[:6]):  # limit to 6
            if key not in history_df.columns:
                continue
            valid_df = history_df[[step_key, key]].dropna()
            if len(valid_df) == 0:
                continue
            x = valid_df[step_key].values
            y = valid_df[key].values
            label = key.split("/")[-1]
            if len(y) > rolling_window:
                y = pd.Series(y).rolling(window=rolling_window, center=True).mean().values
            ax.plot(x, y, color=colors[key_idx % len(colors)], linewidth=1.5, label=label)
        ax.set_xlabel(step_key.replace("_", " ").title())
        ax.set_ylabel("Length")
        ax.set_title("Output Lengths Over Training")
        ax.legend(loc="best", fontsize=8)
        ax.grid(True, alpha=0.3)
        ax_idx += 1
    # plot ratio/boolean metrics
    if ratio_keys:
        ax = axes[ax_idx]
        for key_idx, key in enumerate(ratio_keys[:6]):
            if key not in history_df.columns:
                continue
            valid_df = history_df[[step_key, key]].dropna()
            if len(valid_df) == 0:
                continue
            x = valid_df[step_key].values
            y = valid_df[key].values
            label = key.split("/")[-1]
            if len(y) > rolling_window:
                y = pd.Series(y).rolling(window=rolling_window, center=True).mean().values
            ax.plot(x, y, color=colors[key_idx % len(colors)], linewidth=1.5, label=label)
        ax.set_xlabel(step_key.replace("_", " ").title())
        ax.set_ylabel("Ratio")
        ax.set_title("Parsing Ratios Over Training")
        ax.legend(loc="best", fontsize=8)
        ax.grid(True, alpha=0.3)
        ax_idx += 1
    # plot other metrics
    if other_keys:
        ax = axes[ax_idx]
        for key_idx, key in enumerate(other_keys[:6]):
            if key not in history_df.columns:
                continue
            valid_df = history_df[[step_key, key]].dropna()
            if len(valid_df) == 0:
                continue
            x = valid_df[step_key].values
            y = valid_df[key].values
            label = key.split("/")[-1]
            if len(y) > rolling_window:
                y = pd.Series(y).rolling(window=rolling_window, center=True).mean().values
            ax.plot(x, y, color=colors[key_idx % len(colors)], linewidth=1.5, label=label)
        ax.set_xlabel(step_key.replace("_", " ").title())
        ax.set_ylabel("Value")
        ax.set_title("Other Parsing Metrics")
        ax.legend(loc="best", fontsize=8)
        ax.grid(True, alpha=0.3)
    return fig


fig = plot_parsing_stats(history_df, available_keys["parsing"], step_key=step_key)
plt.tight_layout()
plt.show()

In [ ]:
def extract_parsing_summary(
    summary: dict[str, typing.Any],
) -> dict[str, dict[str, float]]:
    """Extract parsing summary statistics."""
    result: dict[str, dict[str, float]] = {}
    parsing_prefixes = ("parsing/", "reward/run/parsing/")
    for key, value in summary.items():
        if not any(key.startswith(p) for p in parsing_prefixes):
            continue
        if not isinstance(value, (int, float)):
            continue
        # extract metric name and stat
        parts = key.split("/")
        if len(parts) >= 2:
            metric_name = parts[-2] if parts[-1] in ["mean", "std", "min", "max"] else parts[-1]
            stat = parts[-1] if parts[-1] in ["mean", "std", "min", "max"] else "value"
            if metric_name not in result:
                result[metric_name] = {}
            result[metric_name][stat] = float(value)
    return result


def plot_parsing_summary(
    summary: dict[str, typing.Any],
    figsize: tuple[int, int] = (14, 6),
) -> matplotlib.figure.Figure:
    """Bar chart of final parsing statistics."""
    fig, ax = plt.subplots(figsize=figsize)
    parsing_stats = extract_parsing_summary(summary)
    if not parsing_stats:
        ax.text(
            0.5,
            0.5,
            "No parsing summary statistics found",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        ax.set_title("Parsing Summary")
        return fig
    # separate into length vs ratio metrics
    length_metrics = {k: v for k, v in parsing_stats.items() if "length" in k}
    ratio_metrics = {k: v for k, v in parsing_stats.items() if "ratio" in k or "missing" in k or "malformed" in k}
    if length_metrics:
        metrics = list(length_metrics.keys())
        values = [length_metrics[m].get("mean", length_metrics[m].get("value", 0)) for m in metrics]
        x = np.arange(len(metrics))
        bars = ax.bar(x, values, color=plt.cm.Blues(np.linspace(0.4, 0.8, len(metrics))), alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(metrics, rotation=45, ha="right", fontsize=9)
        ax.set_ylabel("Mean Length")
        ax.set_title("Parsing Length Statistics")
        for bar in bars:
            height = bar.get_height()
            ax.annotate(
                f"{height:.1f}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=8,
            )
    elif ratio_metrics:
        metrics = list(ratio_metrics.keys())
        values = [ratio_metrics[m].get("mean", ratio_metrics[m].get("value", 0)) for m in metrics]
        x = np.arange(len(metrics))
        bars = ax.bar(x, values, color=plt.cm.Oranges(np.linspace(0.4, 0.8, len(metrics))), alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(metrics, rotation=45, ha="right", fontsize=9)
        ax.set_ylabel("Ratio")
        ax.set_title("Parsing Ratio Statistics")
        ax.set_ylim(0, 1.1)
        for bar in bars:
            height = bar.get_height()
            ax.annotate(
                f"{height:.2%}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=8,
            )
    ax.grid(axis="y", alpha=0.3)
    return fig


fig = plot_parsing_summary(summary)
plt.tight_layout()
plt.show()

## Training Dashboard

In [ ]:
def plot_training_dashboard(
    history_df: pd.DataFrame,
    summary: dict[str, typing.Any],
    available_keys: dict[str, list[str]],
    step_key: str = "_step",
    figsize: tuple[int, int] = (16, 12),
) -> matplotlib.figure.Figure:
    """Create a summary dashboard with key metrics."""
    fig, axes = plt.subplots(2, 3, figsize=figsize)
    colors = plt.cm.tab10.colors
    # 1. total reward over time
    ax = axes[0, 0]
    if "reward/total" in history_df.columns:
        valid_df = history_df[[step_key, "reward/total"]].dropna()
        x, y = valid_df[step_key].values, valid_df["reward/total"].values
        ax.scatter(x, y, alpha=0.2, s=5, color=colors[0])
        if len(y) > 50:
            rolling = pd.Series(y).rolling(window=50, center=True).mean()
            ax.plot(x, rolling, color="#D7191C", linewidth=2)
    ax.set_title("Total Reward")
    ax.set_xlabel(step_key)
    ax.grid(True, alpha=0.3)
    # 2. reward terms summary
    ax = axes[0, 1]
    term_stats = extract_term_stats(summary)
    if term_stats:
        terms = list(term_stats.keys())[:6]
        means = [term_stats[t].get("mean", 0) for t in terms]
        ax.barh(terms, means, color=colors[: len(terms)])
    ax.set_title("Reward Terms (Mean)")
    ax.grid(axis="x", alpha=0.3)
    # 3. category rewards
    ax = axes[0, 2]
    cat_stats = extract_category_stats(summary)
    if cat_stats:
        cats = list(cat_stats.keys())[:6]
        means = [cat_stats[c].get("mean", 0) for c in cats]
        ax.barh(cats, means, color=plt.cm.viridis(np.linspace(0.3, 0.7, len(cats))))
    ax.set_title("Rewards by Category")
    ax.grid(axis="x", alpha=0.3)
    # 4. completion lengths (TRL)
    ax = axes[1, 0]
    comp_keys = [k for k in available_keys.get("trl", []) if "completions/" in k and "length" in k]
    for key_idx, key in enumerate(comp_keys[:3]):
        if key in history_df.columns:
            valid_df = history_df[[step_key, key]].dropna()
            x, y = valid_df[step_key].values, valid_df[key].values
            if len(y) > 50:
                y = pd.Series(y).rolling(window=50, center=True).mean().values
            ax.plot(x, y, label=key.split("/")[-1], color=colors[key_idx])
    ax.set_title("Completion Lengths")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    # 5. KL divergence
    ax = axes[1, 1]
    if "kl" in history_df.columns:
        valid_df = history_df[[step_key, "kl"]].dropna()
        x, y = valid_df[step_key].values, valid_df["kl"].values
        ax.scatter(x, y, alpha=0.3, s=5, color=colors[1])
        if len(y) > 50:
            rolling = pd.Series(y).rolling(window=50, center=True).mean()
            ax.plot(x, rolling, color="#D7191C", linewidth=2)
    ax.set_title("KL Divergence")
    ax.grid(True, alpha=0.3)
    # 6. entropy
    ax = axes[1, 2]
    if "entropy" in history_df.columns:
        valid_df = history_df[[step_key, "entropy"]].dropna()
        x, y = valid_df[step_key].values, valid_df["entropy"].values
        ax.scatter(x, y, alpha=0.3, s=5, color=colors[2])
        if len(y) > 50:
            rolling = pd.Series(y).rolling(window=50, center=True).mean()
            ax.plot(x, rolling, color="#D7191C", linewidth=2)
    ax.set_title("Policy Entropy")
    ax.grid(True, alpha=0.3)
    fig.suptitle(f"Training Dashboard: {run.name}", fontsize=14, y=1.02)
    return fig


fig = plot_training_dashboard(history_df, summary, available_keys, step_key=step_key)
plt.tight_layout()
plt.show()

## Sample Output Viewer

In [ ]:
def normalize_rewards_table(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize rewards table to consistent schema."""
    df = df.copy()
    json_columns = [
        "reward_terms_json",
        "reward_terms_raw_json",
        "reward_metrics_json",
        "categories_json",
        "tags_json",
    ]
    for col in json_columns:
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
    required_cols = [
        "sample_id",
        "step",
        "prompt",
        "expected_output",
        "model_output",
        "reward_total",
        "generation_idx",
    ]
    for col in required_cols:
        if col not in df.columns:
            df[col] = None
    return df


rewards_table = nb_utils.fetch_wandb_table(run, "reward/rewards_table")
if rewards_table is not None:
    rewards_table = normalize_rewards_table(rewards_table)
    print(f"Loaded {len(rewards_table)} sample records from rewards table")
    print(f"Columns: {list(rewards_table.columns)}")
else:
    print("No rewards table found (log_tables may be disabled in reward logging config)")

In [ ]:
def display_sample(row: pd.Series, show_full: bool = False) -> None:
    """Display a single sample from the rewards table."""
    max_len = 10000 if show_full else 500
    html_parts = []
    html_parts.append(f"<h4>Sample: {row.get('sample_id', 'N/A')} (Step {row.get('step', 'N/A')})</h4>")
    html_parts.append(f"<b>Reward:</b> {row.get('reward_total', 'N/A'):.4f}<br>")
    # generation index
    gen_idx = row.get("generation_idx")
    if gen_idx is not None:
        html_parts.append(f"<b>Generation Index:</b> {gen_idx}<br>")
    # categories
    cats = row.get("categories_json", [])
    if cats:
        html_parts.append(f"<b>Categories:</b> {', '.join(cats) if isinstance(cats, list) else str(cats)}<br>")
    # reward terms
    terms = row.get("reward_terms_json", {})
    if terms and isinstance(terms, dict):
        terms_str = ", ".join(f"{k}: {v:.3f}" for k, v in terms.items())
        html_parts.append(f"<b>Terms:</b> {terms_str}<br>")
    # raw reward terms (if available)
    raw_terms = row.get("reward_terms_raw_json", {})
    if raw_terms and isinstance(raw_terms, dict):
        raw_terms_str = ", ".join(f"{k}: {v:.3f}" for k, v in raw_terms.items())
        html_parts.append(f"<b>Raw Terms:</b> {raw_terms_str}<br>")
    html_parts.append("<hr>")
    # prompt
    prompt, truncated = nb_utils.truncate_text(row.get("prompt"), max_len)
    html_parts.append(f"<b>Prompt:</b>{'[truncated]' if truncated else ''}<br><pre>{prompt}</pre>")
    # expected output
    expected = row.get("expected_output")
    if expected:
        expected_text, truncated = nb_utils.truncate_text(expected, max_len)
        html_parts.append(f"<b>Expected Output:</b>{'[truncated]' if truncated else ''}<br><pre>{expected_text}</pre>")
    # model output
    output, truncated = nb_utils.truncate_text(row.get("model_output"), max_len)
    html_parts.append(f"<b>Model Output:</b>{'[truncated]' if truncated else ''}<br><pre>{output}</pre>")
    # reasoning
    if "reasoning" in row and row.get("reasoning"):
        reasoning, truncated = nb_utils.truncate_text(row.get("reasoning"), max_len)
        html_parts.append(f"<b>Reasoning:</b>{'[truncated]' if truncated else ''}<br><pre>{reasoning}</pre>")
    # final answer
    if "final_answer" in row and row.get("final_answer"):
        answer, truncated = nb_utils.truncate_text(row.get("final_answer"), max_len)
        html_parts.append(f"<b>Final Answer:</b>{'[truncated]' if truncated else ''}<br><pre>{answer}</pre>")
    ipy_display.display(ipy_display.HTML("".join(html_parts)))


if rewards_table is not None and len(rewards_table) > 0:
    print("Displaying first sample:")
    display_sample(rewards_table.iloc[0])
else:
    print("No samples to display")

In [ ]:
# interactive sample browser using ipywidgets
try:
    import ipywidgets

    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("ipywidgets not available - interactive browser disabled")


def create_sample_browser(
    rewards_table: pd.DataFrame,
    run_url: str,
) -> "ipywidgets.VBox":
    """Create an interactive sample browser widget."""
    current_df = rewards_table.copy()
    # filter controls
    show_malformed = ipywidgets.Checkbox(value=False, description="Malformed only")
    show_missing_answer = ipywidgets.Checkbox(value=False, description="Missing answer")
    show_missing_reasoning = ipywidgets.Checkbox(value=False, description="Missing reasoning")
    show_full_text = ipywidgets.Checkbox(value=False, description="Show full text")
    # navigation
    idx_slider = ipywidgets.IntSlider(value=0, min=0, max=max(0, len(current_df) - 1), description="Index:")
    prev_btn = ipywidgets.Button(description="< Prev")
    next_btn = ipywidgets.Button(description="Next >")
    # output area
    output = ipywidgets.Output()
    info_label = ipywidgets.HTML(value=f"Total samples: {len(current_df)}")

    def apply_filters() -> None:
        nonlocal current_df
        df = rewards_table.copy()
        if show_malformed.value and "is_malformed" in df.columns:
            df = df[df["is_malformed"] == True]  # noqa: E712
        if show_missing_answer.value:
            if "final_answer" in df.columns:
                df = df[df["final_answer"].isna() | (df["final_answer"] == "")]
            elif "has_answer" in df.columns:
                df = df[df["has_answer"] == False]  # noqa: E712
        if show_missing_reasoning.value:
            if "reasoning" in df.columns:
                df = df[df["reasoning"].isna() | (df["reasoning"] == "")]
            elif "has_reasoning" in df.columns:
                df = df[df["has_reasoning"] == False]  # noqa: E712
        current_df = df
        idx_slider.max = max(0, len(current_df) - 1)
        idx_slider.value = min(idx_slider.value, idx_slider.max)
        info_label.value = f"Showing {len(current_df)} of {len(rewards_table)} samples"
        update_display(None)

    def update_display(_: typing.Any) -> None:
        output.clear_output()
        with output:
            if len(current_df) == 0:
                print("No samples match current filters")
                return
            idx = idx_slider.value
            if idx < len(current_df):
                display_sample(current_df.iloc[idx], show_full=show_full_text.value)

    def on_prev(_: typing.Any) -> None:
        if idx_slider.value > 0:
            idx_slider.value -= 1

    def on_next(_: typing.Any) -> None:
        if idx_slider.value < idx_slider.max:
            idx_slider.value += 1

    # connect handlers
    idx_slider.observe(update_display, names="value")
    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    show_malformed.observe(lambda _: apply_filters(), names="value")
    show_missing_answer.observe(lambda _: apply_filters(), names="value")
    show_missing_reasoning.observe(lambda _: apply_filters(), names="value")
    show_full_text.observe(update_display, names="value")
    # initial display
    update_display(None)
    # layout
    filter_box = ipywidgets.HBox([show_malformed, show_missing_answer, show_missing_reasoning, show_full_text])
    nav_box = ipywidgets.HBox([prev_btn, idx_slider, next_btn])
    url_html = ipywidgets.HTML(value=f"<a href='{run_url}' target='_blank'>Open in W&B</a>")
    return ipywidgets.VBox([info_label, filter_box, nav_box, url_html, output])


if WIDGETS_AVAILABLE and rewards_table is not None and len(rewards_table) > 0:
    browser = create_sample_browser(rewards_table, run.url)
    ipy_display.display(browser)
elif rewards_table is None:
    print("No rewards table available for browsing")
else:
    print("No samples in rewards table")

In [ ]:
# sample filtering configuration
FILTER_STEP_RANGE = (None, None)  # (min_step, max_step) or None
FILTER_REWARD_RANGE = (None, None)  # (min_reward, max_reward) or None
FILTER_CATEGORIES: list[str] = []  # list of category strings to include (empty = all)


def filter_samples(
    rewards_table: pd.DataFrame,
    step_range: tuple[int | None, int | None] = (None, None),
    reward_range: tuple[float | None, float | None] = (None, None),
    categories: list[str] | None = None,
) -> pd.DataFrame:
    """Filter samples by step, reward, or category."""
    df = rewards_table.copy()
    # step range filter
    if step_range[0] is not None and "step" in df.columns:
        df = df[df["step"] >= step_range[0]]
    if step_range[1] is not None and "step" in df.columns:
        df = df[df["step"] <= step_range[1]]
    # reward range filter
    if reward_range[0] is not None and "reward_total" in df.columns:
        df = df[df["reward_total"] >= reward_range[0]]
    if reward_range[1] is not None and "reward_total" in df.columns:
        df = df[df["reward_total"] <= reward_range[1]]
    # category filter
    if categories and "categories_json" in df.columns:

        def has_category(cats: typing.Any) -> bool:
            if not isinstance(cats, list):
                return False
            return any(c in cats for c in categories)

        df = df[df["categories_json"].apply(has_category)]
    return df


if rewards_table is not None:
    filtered = filter_samples(rewards_table, FILTER_STEP_RANGE, FILTER_REWARD_RANGE, FILTER_CATEGORIES)
    print(f"Filtered to {len(filtered)} samples (from {len(rewards_table)} total)")
else:
    filtered = None
    print("No rewards table to filter")

In [ ]:
def plot_reward_distributions(
    rewards_table: pd.DataFrame,
    figsize: tuple[int, int] = (14, 10),
) -> matplotlib.figure.Figure:
    """Plot reward distributions."""
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    # 1. total reward histogram
    ax = axes[0, 0]
    if "reward_total" in rewards_table.columns:
        rewards = rewards_table["reward_total"].dropna()
        ax.hist(rewards, bins=50, color="#2C7BB6", alpha=0.7, edgecolor="black")
        ax.axvline(rewards.mean(), color="red", linestyle="--", label=f"Mean: {rewards.mean():.3f}")
        ax.set_xlabel("Reward")
        ax.set_ylabel("Count")
        ax.legend()
    ax.set_title("Total Reward Distribution")
    ax.grid(True, alpha=0.3)
    # 2. reward over steps
    ax = axes[0, 1]
    if "step" in rewards_table.columns and "reward_total" in rewards_table.columns:
        ax.scatter(rewards_table["step"], rewards_table["reward_total"], alpha=0.3, s=10)
    ax.set_title("Reward vs Step")
    ax.set_xlabel("Step")
    ax.set_ylabel("Reward")
    ax.grid(True, alpha=0.3)
    # 3. reward terms breakdown (if available)
    ax = axes[1, 0]
    if "reward_terms_json" in rewards_table.columns:
        # extract term values
        term_data: dict[str, list[float]] = {}
        for terms in rewards_table["reward_terms_json"].dropna():
            if isinstance(terms, dict):
                for term, value in terms.items():
                    if term not in term_data:
                        term_data[term] = []
                    term_data[term].append(value)
        if term_data:
            term_names = list(term_data.keys())
            term_values = [term_data[t] for t in term_names]
            ax.boxplot(term_values, labels=term_names)
            ax.set_xticklabels(term_names, rotation=45, ha="right")
    ax.set_title("Reward Terms Distribution")
    ax.set_ylabel("Value")
    ax.grid(True, alpha=0.3)
    # 4. rewards by category (if available)
    ax = axes[1, 1]
    if "categories_json" in rewards_table.columns and "reward_total" in rewards_table.columns:
        # group by first category
        cat_rewards: dict[str, list[float]] = {}
        for _idx, row in rewards_table.iterrows():
            cats = row.get("categories_json", [])
            reward = row.get("reward_total")
            if isinstance(cats, list) and cats and reward is not None:
                cat = cats[0]  # use first category
                if cat not in cat_rewards:
                    cat_rewards[cat] = []
                cat_rewards[cat].append(reward)
        if cat_rewards:
            cat_names = list(cat_rewards.keys())[:8]  # limit to 8
            cat_values = [cat_rewards[c] for c in cat_names]
            ax.boxplot(cat_values, labels=cat_names)
            ax.set_xticklabels(cat_names, rotation=45, ha="right", fontsize=8)
    ax.set_title("Rewards by Category")
    ax.set_ylabel("Reward")
    ax.grid(True, alpha=0.3)
    return fig


if rewards_table is not None and len(rewards_table) > 0:
    fig = plot_reward_distributions(rewards_table)
    plt.tight_layout()
    plt.show()
else:
    print("No rewards table available for distribution analysis")

In [ ]:
def plot_length_vs_reward(
    rewards_table: pd.DataFrame,
    figsize: tuple[int, int] = (14, 5),
) -> matplotlib.figure.Figure:
    """Scatter plot: output/reasoning length vs reward."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    # compute lengths if not already present
    df = rewards_table.copy()
    if "output_length" not in df.columns and "model_output" in df.columns:
        df["output_length"] = df["model_output"].apply(lambda x: len(str(x)) if x else 0)
    if "reasoning_length" not in df.columns and "reasoning" in df.columns:
        df["reasoning_length"] = df["reasoning"].apply(lambda x: len(str(x)) if x else 0)
    # output length vs reward
    ax = axes[0]
    if "output_length" in df.columns and "reward_total" in df.columns:
        valid = df[["output_length", "reward_total"]].dropna()
        ax.scatter(valid["output_length"], valid["reward_total"], alpha=0.3, s=10, c="#2C7BB6")
        # add trend line
        if len(valid) > 10:
            z = np.polyfit(valid["output_length"], valid["reward_total"], 1)
            p = np.poly1d(z)
            x_line = np.linspace(valid["output_length"].min(), valid["output_length"].max(), 100)
            ax.plot(x_line, p(x_line), "r--", alpha=0.7, label="Trend")
            ax.legend()
    ax.set_xlabel("Output Length (chars)")
    ax.set_ylabel("Reward")
    ax.set_title("Output Length vs Reward")
    ax.grid(True, alpha=0.3)
    # reasoning length vs reward
    ax = axes[1]
    if "reasoning_length" in df.columns and "reward_total" in df.columns:
        valid = df[["reasoning_length", "reward_total"]].dropna()
        valid = valid[valid["reasoning_length"] > 0]  # only samples with reasoning
        if len(valid) > 0:
            ax.scatter(valid["reasoning_length"], valid["reward_total"], alpha=0.3, s=10, c="#7570B3")
            if len(valid) > 10:
                z = np.polyfit(valid["reasoning_length"], valid["reward_total"], 1)
                p = np.poly1d(z)
                x_line = np.linspace(valid["reasoning_length"].min(), valid["reasoning_length"].max(), 100)
                ax.plot(x_line, p(x_line), "r--", alpha=0.7, label="Trend")
                ax.legend()
        else:
            ax.text(
                0.5,
                0.5,
                "No samples with reasoning",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
    ax.set_xlabel("Reasoning Length (chars)")
    ax.set_ylabel("Reward")
    ax.set_title("Reasoning Length vs Reward")
    ax.grid(True, alpha=0.3)
    return fig


if rewards_table is not None and len(rewards_table) > 0:
    fig = plot_length_vs_reward(rewards_table)
    plt.tight_layout()
    plt.show()
else:
    print("No rewards table available for length analysis")

## TRL Policy Metrics (Secondary)

In [ ]:
def plot_completion_lengths(
    history_df: pd.DataFrame,
    trl_keys: list[str],
    step_key: str = "_step",
    figsize: tuple[int, int] = (14, 5),
    rolling_window: int = 50,
) -> matplotlib.figure.Figure:
    """Plot completion length metrics over time."""
    length_keys = [k for k in trl_keys if "completions/" in k and "length" in k]
    clip_keys = [k for k in trl_keys if "clipped" in k]
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    colors = plt.cm.tab10.colors
    # completion lengths
    ax = axes[0]
    for idx, key in enumerate(length_keys[:4]):
        if key not in history_df.columns:
            continue
        valid_df = history_df[[step_key, key]].dropna()
        if len(valid_df) == 0:
            continue
        x = valid_df[step_key].values
        y = valid_df[key].values
        label = key.split("/")[-1]
        if len(y) > rolling_window:
            y = pd.Series(y).rolling(window=rolling_window, center=True).mean().values
        ax.plot(x, y, color=colors[idx], linewidth=1.5, label=label)
    ax.set_xlabel(step_key.replace("_", " ").title())
    ax.set_ylabel("Length (tokens)")
    ax.set_title("Completion Lengths Over Training")
    ax.legend(loc="best", fontsize=8)
    ax.grid(True, alpha=0.3)
    # clipped ratio
    ax = axes[1]
    for idx, key in enumerate(clip_keys[:2]):
        if key not in history_df.columns:
            continue
        valid_df = history_df[[step_key, key]].dropna()
        if len(valid_df) == 0:
            continue
        x = valid_df[step_key].values
        y = valid_df[key].values
        label = key.split("/")[-1]
        if len(y) > rolling_window:
            y = pd.Series(y).rolling(window=rolling_window, center=True).mean().values
        ax.plot(x, y, color=colors[idx], linewidth=1.5, label=label)
    ax.set_xlabel(step_key.replace("_", " ").title())
    ax.set_ylabel("Ratio")
    ax.set_title("Completion Clipped Ratio")
    ax.legend(loc="best", fontsize=8)
    ax.grid(True, alpha=0.3)
    return fig


fig = plot_completion_lengths(history_df, available_keys["trl"], step_key=step_key)
plt.tight_layout()
plt.show()

In [ ]:
def plot_policy_stability(
    history_df: pd.DataFrame,
    step_key: str = "_step",
    figsize: tuple[int, int] = (14, 5),
    rolling_window: int = 50,
) -> matplotlib.figure.Figure:
    """Plot KL divergence, entropy, and clip ratios."""
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    colors = plt.cm.tab10.colors
    # KL divergence
    ax = axes[0]
    if "kl" in history_df.columns:
        valid_df = history_df[[step_key, "kl"]].dropna()
        if len(valid_df) > 0:
            x = valid_df[step_key].values
            y = valid_df["kl"].values
            ax.scatter(x, y, alpha=0.2, s=5, color=colors[0])
            if len(y) > rolling_window:
                rolling = pd.Series(y).rolling(window=rolling_window, center=True).mean()
                ax.plot(x, rolling, color="#D7191C", linewidth=2, label="Rolling mean")
                ax.legend(fontsize=8)
    else:
        ax.text(
            0.5,
            0.5,
            "KL not logged",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
    ax.set_xlabel(step_key.replace("_", " ").title())
    ax.set_ylabel("KL Divergence")
    ax.set_title("KL Divergence (Policy vs Reference)")
    ax.grid(True, alpha=0.3)
    # entropy
    ax = axes[1]
    if "entropy" in history_df.columns:
        valid_df = history_df[[step_key, "entropy"]].dropna()
        if len(valid_df) > 0:
            x = valid_df[step_key].values
            y = valid_df["entropy"].values
            ax.scatter(x, y, alpha=0.2, s=5, color=colors[1])
            if len(y) > rolling_window:
                rolling = pd.Series(y).rolling(window=rolling_window, center=True).mean()
                ax.plot(x, rolling, color="#D7191C", linewidth=2, label="Rolling mean")
                ax.legend(fontsize=8)
    else:
        ax.text(
            0.5,
            0.5,
            "Entropy not logged",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
    ax.set_xlabel(step_key.replace("_", " ").title())
    ax.set_ylabel("Entropy")
    ax.set_title("Policy Entropy (Exploration)")
    ax.grid(True, alpha=0.3)
    # clip ratio
    ax = axes[2]
    clip_keys = [k for k in history_df.columns if "clip_ratio" in k]
    if clip_keys:
        for idx, key in enumerate(clip_keys[:3]):
            valid_df = history_df[[step_key, key]].dropna()
            if len(valid_df) == 0:
                continue
            x = valid_df[step_key].values
            y = valid_df[key].values
            label = key.split("/")[-1] if "/" in key else key
            if len(y) > rolling_window:
                y = pd.Series(y).rolling(window=rolling_window, center=True).mean().values
            ax.plot(x, y, color=colors[idx], linewidth=1.5, label=label)
        ax.legend(fontsize=8)
    else:
        ax.text(
            0.5,
            0.5,
            "Clip ratio not logged",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
    ax.set_xlabel(step_key.replace("_", " ").title())
    ax.set_ylabel("Clip Ratio")
    ax.set_title("Policy Update Clipping")
    ax.grid(True, alpha=0.3)
    return fig


fig = plot_policy_stability(history_df, step_key=step_key)
plt.tight_layout()
plt.show()

## Self-Check

In [ ]:
def run_notebook_checks() -> None:
    """Verify notebook loaded data correctly and produced outputs."""
    checks = {
        "Run loaded": run.name if "run" in dir() else "N/A",
        "Run state": run.state if "run" in dir() else "N/A",
        "History rows loaded": len(history_df) if "history_df" in dir() else 0,
        "History columns": len(history_df.columns) if "history_df" in dir() else 0,
        "Step key": step_key if "step_key" in dir() else "N/A",
        "Reward total keys": len(available_keys.get("reward_total", [])) if "available_keys" in dir() else 0,
        "Reward term keys": len(available_keys.get("reward_terms", [])) if "available_keys" in dir() else 0,
        "Parsing keys": len(available_keys.get("parsing", [])) if "available_keys" in dir() else 0,
        "TRL keys": len(available_keys.get("trl", [])) if "available_keys" in dir() else 0,
        "Rewards table": len(rewards_table) if "rewards_table" in dir() and rewards_table is not None else "Not logged",
    }
    print("Notebook Self-Check:")
    print("=" * 50)
    for check, value in checks.items():
        if isinstance(value, int):
            status = "OK" if value > 0 else "WARN"
        elif value in ["N/A", "Not logged"]:
            status = "WARN"
        else:
            status = "OK"
        print(f"  [{status:4}] {check}: {value}")
    print("=" * 50)
    # summary
    warnings = sum(1 for v in checks.values() if v in ["N/A", "Not logged", 0])
    if warnings == 0:
        print("All checks passed!")
    else:
        print(f"{warnings} warning(s) - some data may not be available")


run_notebook_checks()